In [ ]:
import sys
sys.path.append("/home/149/ab8992/tasman-tides/")
import xarray as xr
import ttidelib as tt
import scipy
import cmocean
import os
from pathlib import Path
cmap = cmocean.cm.dense_r
import matplotlib
import matplotlib.pyplot as plt
earth_cmap = matplotlib.colormaps["gist_earth"]
from datetime import timedelta
import warnings
warnings.simplefilter("ignore")
# import filtering
import numpy as np
import dask
dask.config.set({'logging.distributed': 'error'})
from dask.distributed import Client,default_client
import xrft


client = tt.startdask(nthreads=1,n_workers = 40)

client

<Client: 'tcp://127.0.0.1:46185' processes=40 threads=40, memory=416.00 GiB>


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 40
Total threads: 40,Total memory: 416.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46185,Workers: 40
Dashboard: /proxy/8787/status,Total threads: 40
Started: Just now,Total memory: 416.00 GiB
Comm: tcp://127.0.0.1:38705,Total threads: 1
Dashboard: /proxy/42323/status,Memory: 10.40 GiB
Nanny: tcp://127.0.0.1:40655,


2025-10-08 12:31:21,454 - distributed.semaphore - WARNING - Tried to release Lock or Semaphore but it was already released: name='/scratch/nm03/ab8992/forcallum/outputs/full-40/u_0.nc', lease_id='5bfe681a8d8a400693a9fffc6d2fd0e7'. This can happen if the Lock or Semaphore timed out before.
2025-10-08 12:50:48,807 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:38917 (pid=719185) exceeded 95% memory budget. Restarting...
2025-10-08 12:50:48,897 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:38917' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('getitem-a200a08202fbc23e0bf765ab4b33ead4', 0, 0, 0, 0, 0), ('getitem-d008769f316aa1a44bc2c42f9b4bc8f0', 0, 0, 0, 0, 0)} (stimulus_id='handle-worker-cleanup-1759888248.897433')
2025-10-08 12:50:48,983 - distributed.nanny - WARNING - Restarting worker
2025-10-08 12:50:51,952 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:42023 (pid=719075) exceeded 95% mem

In [14]:
tmpstorage = os.getenv('PBS_JOBFS')
tmpstorage = "/scratch/nm03/ab8992/test"
outputdir = "/scratch/nm03/ab8992/test/outputs"
def DirectionalFilter(data,dim = "xb"):
    """
    Fourier filter into forward and backward propagating signals
    """

    
    if "lat" in data.coords:
        data = data.drop(["lat","lon"])
    import xrft
    FT = xrft.fft(
        data,dim = ["time",dim]
    )

    forward = np.real(xrft.ifft(
        FT.where((FT[f"freq_{dim}"] > 0) & (FT.freq_time > 0), 0) +
          FT.where((FT[f"freq_{dim}"] < 0) & (FT.freq_time < 0), 0),
        dim = ["freq_time",f"freq_{dim}"]
    ))

    backward = np.real(xrft.ifft(
        FT.where((FT[f"freq_{dim}"] < 0) & (FT.freq_time > 0), 0) + 
        FT.where((FT[f"freq_{dim}"] > 0) & (FT.freq_time < 0), 0),
        dim = ["freq_time",f"freq_{dim}"]
    ))

    return xr.merge([forward.rename(f"{data.name}_forward"),backward.rename(f"{data.name}_backward")]).assign_coords({"time":data.time,dim:data[dim]})
def laplacian(data):
    out = data.u.bfill("xb",limit = 4).ffill("xb",limit = 4).differentiate("xb").differentiate("xb")
    out +=data.v.bfill("yb",limit = 4).ffill("yb",limit = 4).differentiate("yb").differentiate("yb")
    return out

def _scipy_integrate(data, zl):
    """
    Helper function to perform cumulative trapezoidal integration along the 'zl' axis.
    """
    return scipy.integrate.cumulative_trapezoid(data, x=zl, initial=0)

def scipy_integrate(data):
    """
    Wrapper function to apply cumulative trapezoidal integration along the 'zl' axis.
    """
    zl = data.zl.values  # Extract 'zl' coordinate values
    return np.apply_along_axis(_scipy_integrate, data.get_axis_num('zl'), data, zl=zl)

def save_temporary(expt,t0,outputdir):
    """
    Open all the datasets that we need for this experiment. Save some of the simple stuff, and save the modal decompossitions to temporary storage chunked by mode for further processing.
    """
    with (
            xr.open_mfdataset(f"/g/data/nm03/ab8992/postprocessed/figdata/snapshots/{expt}/u*.nc",decode_times = False) as uamp,
            xr.open_mfdataset(f"/g/data/nm03/ab8992/postprocessed/figdata/snapshots/{expt}/v*.nc",decode_times = False) as vamp
    ):
        if "80" in expt: #! vmodes have failed for some times. For now, fix vmodes to match for each run
            vmodes= xr.open_dataset(f"/g/data/nm03/ab8992/postprocessed/{expt}/vertical_eigenfunctions/vmode-t0-{12258}.nc",decode_times = False,chunks = {"mode":1}).isel(zl = slice(0,96))
            vmodes = vmodes.interp_like(uamp.u.isel(time = 0)).persist() ## Now need to chunk this on disk like the Filtered data
            print("vmodes interpolated")
            ## Fix messed up times:
        elif "40" in expt: #! vmodes have failed for some times. For now, fix vmodes to match for each run
            vmodes = xr.open_dataset(f"/g/data/nm03/ab8992/postprocessed/{expt}/vertical_eigenfunctions/vmode-t0-{4216}.nc",decode_times = False,chunks = {"mode":1}).isel(zl = slice(0,96))
        else:
            vmodes = xr.open_dataset(f"/g/data/nm03/ab8992/postprocessed/{expt}/vertical_eigenfunctions/vmode-t0-{22000}.nc",decode_times = False,chunks = {"mode":1}).isel(zl = slice(0,96))

            
        # os.remove(f"{tmpstorage}/*.nc")
        # vmodes = vmodes.assign_coords({"zl":filtered.zl})
        ymin = vmodes.yb[0].values - 0.0001 ## This ensures random numerical changes of 1e-16 to axis values don't cause issues... 
        ymax = vmodes.yb[-1].values + 0.0001


        ## Here all variables are dask arrays. One by one, compute what we want and save to PBS temporart storage 

        for i in range(len(vmodes.mode.values)):
            # print("Saving mode ",i)
            u = (uamp.u.isel(mode = [i]) * vmodes.U.isel(mode = [i])).rename("u").to_netcdf(f"{outputdir}/u_{i}.nc")
            v = (vamp.v.isel(mode = [i]) * vmodes.V.isel(mode = [i])).rename("v").to_netcdf(f"{outputdir}/v_{i}.nc")

        # Get N
        # data = tt.collect_data(
        #     exptname=expt,
        #     rawdata = ["rho"],
        #     timerange = (t0 - 233,t0 + 233 )
        # ).sel(yb = slice(-125,125))
        # if "zi" in data:
        #     data = data.drop_vars("zi")
        # H = data.bathy
        # if H.mean("xb").mean("yb") <= 0:
        #     H *= -1
        # # tt.logmsg("VMODES: data loaded.")
        # N = tt.getN(data.rho).mean("time")
        # N.to_netcdf(f"{outputdir}/N.nc")

In [15]:
import shutil
expts = ["full-10","beamless-10","smooth-10","beamless-20","smooth-20","full-20","full-40","beamless-40","smooth-40"]
expts = ["full-40","beamless-40","smooth-40"]
# expts = ["full-40","beamless-40","smooth-40"]
# expts = ["full-10"]
t0_20th = 22000
t0_40th = 4216
t0_80th = 12023
for expt in expts:
    outputdir = "/scratch/nm03/ab8992/forcallum/outputs/"+expt
    tmpstorage = "/scratch/nm03/ab8992/forcallum/tmpstorage/"+expt
    t0 = t0_20th
    if "40" in expt:
        t0 = t0_40th
    if not os.path.exists(outputdir):
        os.makedirs(outputdir)
    if not os.path.exists(tmpstorage):
        os.makedirs(tmpstorage)
    print(expt)

    save_temporary(expt,t0,outputdir)
    # save_modal(outputdir)
    print("Done with ",expt)
    # clear_output()

full-40


2025-10-08 12:53:23,278 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 7.74 GiB -- Worker memory limit: 10.40 GiB
2025-10-08 12:53:23,666 - distributed.worker.memory - WARNING - Worker is at 82% memory usage. Pausing worker.  Process memory: 8.53 GiB -- Worker memory limit: 10.40 GiB
2025-10-08 12:53:26,713 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 7.66 GiB -- Worker memory limit: 10.40 GiB
2025-10-08 12:53:27,141 - distributed.worker.memory - WARNING - Worker is at 80% memory usage. Pausing worker.  Process m

KilledWorker: Attempted to run task ('mul-store-map-6d94723bfd0cfc73518b3887c8ae0856', 0, 0, 0, 0, 0) on 4 different workers, but all those workers died while running it. The last worker that attempt to run the task was tcp://127.0.0.1:43717. Inspecting worker logs is often a good next step to diagnose what went wrong. For more information see https://distributed.dask.org/en/stable/killed.html.

In [11]:
uamp = xr.open_dataset("/g/data/nm03/ab8992/postprocessed/figdata/snapshots/full-20/u_1.nc")
vmode = xr.open_dataset("/g/data/nm03/ab8992/postprocessed/full-20/vertical_eigenfunctions/vmode-t0-22000.nc")


In [21]:
vmode.U.yb.values

array([-120., -116., -112., -108., -104., -100.,  -96.,  -92.,  -88.,
        -84.,  -80.,  -76.,  -72.,  -68.,  -64.,  -60.,  -56.,  -52.,
        -48.,  -44.,  -40.,  -36.,  -32.,  -28.,  -24.,  -20.,  -16.,
        -12.,   -8.,   -4.,    0.,    4.,    8.,   12.,   16.,   20.,
         24.,   28.,   32.,   36.,   40.,   44.,   48.,   52.,   56.,
         60.,   64.,   68.,   72.,   76.,   80.,   84.,   88.,   92.,
         96.,  100.,  104.,  108.,  112.,  116.,  120.])

In [20]:
vmode.U.isel(mode = [0]).yb.values

array([-120., -116., -112., -108., -104., -100.,  -96.,  -92.,  -88.,
        -84.,  -80.,  -76.,  -72.,  -68.,  -64.,  -60.,  -56.,  -52.,
        -48.,  -44.,  -40.,  -36.,  -32.,  -28.,  -24.,  -20.,  -16.,
        -12.,   -8.,   -4.,    0.,    4.,    8.,   12.,   16.,   20.,
         24.,   28.,   32.,   36.,   40.,   44.,   48.,   52.,   56.,
         60.,   64.,   68.,   72.,   76.,   80.,   84.,   88.,   92.,
         96.,  100.,  104.,  108.,  112.,  116.,  120.])

In [24]:
a = uamp.u.isel(mode = [0])
b = vmode.U.isel(mode = [0])

In [ ]:
b.

<xarray.DataArray 'U' (mode: 1, zl: 100, yb: 61, xb: 376)>
[2293600 values with dtype=float64]
Coordinates:
  * mode     (mode) int64 0
  * zl       (zl) float64 2.704 8.126 13.58 ... 5.236e+03 5.339e+03 5.443e+03
  * xb       (xb) float64 -0.0 4.0 8.0 12.0 ... 1.492e+03 1.496e+03 1.5e+03
  * yb       (yb) float64 -120.0 -116.0 -112.0 -108.0 ... 112.0 116.0 120.0

In [29]:
a

<xarray.DataArray 'u' (mode: 1, yb: 61, xb: 376, time: 233)>
[5344088 values with dtype=float64]
Coordinates:
  * mode     (mode) int64 1
  * xb       (xb) float64 -0.0 4.0 8.0 12.0 ... 1.492e+03 1.496e+03 1.5e+03
  * yb       (yb) float64 -120.0 -116.0 -112.0 -108.0 ... 112.0 116.0 120.0
  * time     (time) object 2012-06-30 19:00:00 ... 2012-07-10 11:00:00

In [ ]:
a * b